In [2]:
!pip install ultralytics opencv-python pandas matplotlib -q

In [4]:
from dataclasses import dataclass
from datetime import date, datetime, timedelta
from collections import defaultdict

# Путь к весам детектора. Пока стоит предобученный YOLO (классы COCO) —
# он реально видит только "truck"/"car". После дообучения на датасете
# техники сюда подставляется путь к своим весам (best.pt).
DETECTOR_WEIGHTS = "yolov8m.pt"

EQUIPMENT_CLASSES = ["excavator", "crane", "bulldozer", "truck", "loader", "concrete_mixer", "roller"]

EQUIPMENT_TO_WORK_TYPE = {
    "excavator": ["earthworks", "foundation"],
    "bulldozer": ["earthworks", "site_preparation"],
    "crane": ["structural_assembly", "installation"],
    "truck": ["material_delivery", "earthworks"],
    "loader": ["earthworks", "material_handling"],
    "concrete_mixer": ["concrete_works"],
    "roller": ["road_works", "site_preparation"],
}

CONF_THRESHOLD = 0.35
TRACKER_CONFIG = "bytetrack.yaml"
MIN_PRESENCE_SECONDS = 120  # порог, чтобы не считать случайный проезд "работой"

In [5]:
from matplotlib.path import Path as MplPath

# zone_id -> (название, полигон [[x,y], ...])
ZONES = {
    "zone_a": ("Котлован / земляные работы", [[0, 0], [640, 0], [640, 360], [0, 360]]),
    "zone_b": ("Монтажная площадка",          [[640, 0], [1280, 0], [1280, 360], [640, 360]]),
    "zone_c": ("Подъездные пути",             [[0, 360], [1280, 360], [1280, 720], [0, 720]]),
}
_zone_paths = {zid: MplPath(poly) for zid, (name, poly) in ZONES.items()}

def assign_zone(point_xy):
    for zid, path in _zone_paths.items():
        if path.contains_point(point_xy):
            return zid
    return None

In [6]:
SCHEDULE = [
    {"stage_id": "1", "work_type": "site_preparation", "zone_id": "zone_a",
     "start": date(2026, 9, 1), "end": date(2026, 9, 5),
     "expected_equipment": {"bulldozer", "truck"}, "min_count": 1, "max_count": 3},
    {"stage_id": "2", "work_type": "earthworks", "zone_id": "zone_a",
     "start": date(2026, 9, 6), "end": date(2026, 9, 15),
     "expected_equipment": {"excavator", "truck"}, "min_count": 1, "max_count": 4},
    {"stage_id": "4", "work_type": "structural_assembly", "zone_id": "zone_b",
     "start": date(2026, 9, 10), "end": date(2026, 9, 30),
     "expected_equipment": {"crane", "truck"}, "min_count": 1, "max_count": 2},
]

def active_stages(day, zone_id=None):
    result = [s for s in SCHEDULE if s["start"] <= day <= s["end"]]
    if zone_id is not None:
        result = [s for s in result if s["zone_id"] == zone_id]
    return result

In [7]:
from ultralytics import YOLO
import cv2

model = YOLO(DETECTOR_WEIGHTS)

def run_detection(video_path, video_start_time, sample_every_n_frames=1):
    """Генерирует детекции: (timestamp, track_id, class_name, center_xy)."""
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    cap.release()

    stream = model.track(source=video_path, conf=CONF_THRESHOLD,
                          tracker=TRACKER_CONFIG, stream=True, verbose=False)

    for frame_idx, result in enumerate(stream):
        if frame_idx % sample_every_n_frames != 0:
            continue
        timestamp = video_start_time + timedelta(seconds=frame_idx / fps)
        if result.boxes is None:
            continue
        for box in result.boxes:
            class_name = model.names[int(box.cls[0])]
            if class_name not in EQUIPMENT_CLASSES:
                continue
            track_id = int(box.id[0]) if box.id is not None else None
            x1, y1, x2, y2 = box.xyxy[0].tolist()
            yield {
                "timestamp": timestamp,
                "track_id": track_id,
                "class_name": class_name,
                "center_xy": ((x1 + x2) / 2, (y1 + y2) / 2),
            }

Creating new Ultralytics Settings v0.0.8 file  
View Ultralytics Settings with 'yolo settings' or at 'C:\Users\edwar\AppData\Roaming\Ultralytics\settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.


In [9]:
def aggregate_observations(detections):
    # (zone_id, day) -> {"tracks_by_class": {...}, "presence_seconds": {...}}
    obs = {}
    last_seen = {}

    for det in detections:
        if det["track_id"] is None:
            continue
        zone_id = assign_zone(det["center_xy"])
        if zone_id is None:
            continue
        day = det["timestamp"].date()
        key = (zone_id, day)
        if key not in obs:
            obs[key] = {"tracks_by_class": defaultdict(set), "presence_seconds": defaultdict(float)}
        obs[key]["tracks_by_class"][det["class_name"]].add(det["track_id"])

        last_key = (zone_id, det["class_name"], det["track_id"])
        if last_key in last_seen:
            delta = (det["timestamp"] - last_seen[last_key]).total_seconds()
            if 0 < delta < 300:
                obs[key]["presence_seconds"][(det["class_name"], det["track_id"])] += delta
        last_seen[last_key] = det["timestamp"]

    return obs

def equipment_counts(obs_entry):
    counts = {}
    for cls, track_ids in obs_entry["tracks_by_class"].items():
        valid = {tid for tid in track_ids
                 if obs_entry["presence_seconds"].get((cls, tid), 0) >= MIN_PRESENCE_SECONDS}
        if valid:
            counts[cls] = len(valid)
    return counts

In [10]:
def find_deviations(observations):
    deviations = []
    for (zone_id, day), entry in observations.items():
        observed = equipment_counts(entry)
        stages = active_stages(day, zone_id)

        if not stages:
            if observed:
                deviations.append({"day": day, "zone_id": zone_id, "kind": "no_scheduled_activity",
                                    "detail": f"Техника {observed} есть, но работ по графику нет"})
            continue

        for stage in stages:
            expected = stage["expected_equipment"]
            observed_set = set(observed.keys())

            missing = expected - observed_set
            if missing:
                deviations.append({"day": day, "zone_id": zone_id, "stage_id": stage["stage_id"],
                                    "kind": "missing_equipment",
                                    "detail": f"Не хватает техники: {missing}"})

            unexpected = observed_set - expected
            if unexpected:
                deviations.append({"day": day, "zone_id": zone_id, "stage_id": stage["stage_id"],
                                    "kind": "unexpected_equipment",
                                    "detail": f"Техника вне плана: {unexpected}"})

            total = sum(n for cls, n in observed.items() if cls in expected)
            if expected and not (stage["min_count"] <= total <= stage["max_count"]):
                deviations.append({"day": day, "zone_id": zone_id, "stage_id": stage["stage_id"],
                                    "kind": "count_out_of_range",
                                    "detail": f"Ожидалось {stage['min_count']}-{stage['max_count']}, факт {total}"})
    return deviations

In [14]:
detections = list(run_detection("your_video.mp4", datetime(2026, 9, 5, 8, 0, 0)))
observations = aggregate_observations(detections)
deviations = find_deviations(observations)

import pandas as pd
pd.DataFrame(deviations)

,day,zone_id,kind,detail,stage_id
0,2026-09-05,zone_b,no_scheduled_activity,"Техника {'truck': 3} есть, но работ по графику...",NaN
1,2026-09-05,zone_c,no_scheduled_activity,"Техника {'truck': 7} есть, но работ по графику...",NaN
2,2026-09-05,zone_a,missing_equipment,"Не хватает техники: {'bulldozer', 'truck'}",1
3,2026-09-05,zone_a,count_out_of_range,"Ожидалось 1-3, факт 0",1


In [13]:
# 1. Узнаём реальное разрешение видео
import cv2
cap = cv2.VideoCapture("your_video.mp4")
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
cap.release()
print("Разрешение видео:", width, height)

# 2. Пересчитываем зоны под реальное разрешение (пример — те же 3 области, но под 3840x2160)
ZONES = {
    "zone_a": ("Котлован / земляные работы", [[0, 0], [width//2, 0], [width//2, height//2], [0, height//2]]),
    "zone_b": ("Монтажная площадка",          [[width//2, 0], [width, 0], [width, height//2], [width//2, height//2]]),
    "zone_c": ("Подъездные пути",             [[0, height//2], [width, height//2], [width, height], [0, height]]),
}
_zone_paths = {zid: MplPath(poly) for zid, (name, poly) in ZONES.items()}

# 3. Для теста на коротком ролике снижаем порог присутствия до 1 секунды
MIN_PRESENCE_SECONDS = 1

Разрешение видео: 3840 2160
